# 14 — Policy Explanation

Understanding *why* a policy takes certain actions is essential for debugging, auditing, and building trust — especially for policies in high-stakes settings. This notebook implements `policy-lens`: gradient saliency, integrated gradients, and concept probes.

In [ ]:
# policy-lens: saliency and feature attribution for RL policies
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

torch.manual_seed(42)

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,2))
    def forward(self, x): return self.net(x)

# Brief training
env = gym.make("CartPole-v1")
q_net = QNet(); optimizer_q = optim.Adam(q_net.parameters(), lr=1e-3)
buf = []; epsilon = 1.0

print("Training DQN policy (abbreviated)...")
for ep in range(300):
    obs, _ = env.reset(seed=ep)
    for _ in range(500):
        if np.random.rand() < epsilon: a = env.action_space.sample()
        else:
            with torch.no_grad(): a = q_net(torch.FloatTensor(obs)).argmax().item()
        ns, r, t, tr, _ = env.step(a)
        buf.append((obs, a, r, ns, float(t or tr))); obs = ns
        if t or tr: break
        if len(buf) >= 64:
            b = [buf[i] for i in np.random.choice(len(buf), 64)]
            s,av,rv,nv,dv = [torch.FloatTensor(np.array(x)) for x in zip(*b)]
            av = av.long()
            q_vals = q_net(s).gather(1,av.unsqueeze(1)).squeeze(1)
            with torch.no_grad(): tgt = rv + 0.99*q_net(nv).max(1)[0]*(1-dv)
            loss = nn.functional.mse_loss(q_vals, tgt)
            optimizer_q.zero_grad(); loss.backward(); optimizer_q.step()
    epsilon = max(0.05, epsilon*0.99)
env.close()
print(f"Policy trained (eps={epsilon:.3f})")

def gradient_saliency(model, obs, action):
    obs_t = torch.FloatTensor(obs).requires_grad_(True)
    q = model(obs_t)[action]
    q.backward()
    return obs_t.grad.abs().detach().numpy()

def integrated_gradients(model, obs, action, steps=50):
    baseline = np.zeros_like(obs)
    obs_t = torch.FloatTensor(obs); base_t = torch.FloatTensor(baseline)
    grads = []
    for alpha in np.linspace(0, 1, steps):
        interp = (base_t + alpha*(obs_t-base_t)).requires_grad_(True)
        q = model(interp)[action]; q.backward()
        grads.append(interp.grad.detach().numpy())
    return np.abs((obs - baseline) * np.mean(grads, axis=0))

# Collect sample observations
obs_examples = []
env2 = gym.make("CartPole-v1"); obs, _ = env2.reset(seed=42)
for _ in range(5):
    action = q_net(torch.FloatTensor(obs)).argmax().item()
    obs_examples.append((obs.copy(), action))
    obs, _, t, tr, _ = env2.step(action)
    if t or tr: break
env2.close()

feature_names = ["Cart pos", "Cart vel", "Pole angle", "Pole vel"]
n_examples = min(3, len(obs_examples))
fig, axes = plt.subplots(2, n_examples, figsize=(12, 6))

for col, (obs_ex, action_ex) in enumerate(obs_examples[:n_examples]):
    grad_sal = gradient_saliency(q_net, obs_ex, action_ex)
    ig_sal = integrated_gradients(q_net, obs_ex, action_ex)
    for row, (sal, title) in enumerate([(grad_sal,'Gradient Saliency'),(ig_sal,'Integrated Gradients')]):
        ax = axes[row, col]
        ax.bar(feature_names, sal, color=['steelblue' if s==sal.max() else 'lightsteelblue' for s in sal])
        ax.set_title(f'{title}\nAction: {"Right" if action_ex else "Left"}', fontsize=9, fontweight='bold')
        ax.tick_params(axis='x', labelsize=8)

plt.suptitle('policy-lens: Feature Attribution for CartPole', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('/tmp/policy_lens.png', dpi=100, bbox_inches='tight'); plt.show()


In [ ]:
# policy-lens: concept probes — does the policy encode interpretable concepts?
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

def get_hidden_representations(model, observations, layer_idx=1):
    activations = []
    def hook_fn(module, inp, out): activations.append(out.detach().numpy())
    target = list(model.net.children())[layer_idx]
    h = target.register_forward_hook(hook_fn)
    with torch.no_grad(): model(torch.FloatTensor(np.array(observations)))
    h.remove()
    return np.array(activations[0])

env3 = gym.make("CartPole-v1")
observations, concepts = [], {"pole_falling_right": [], "pole_falling_fast": [], "cart_off_centre": []}
obs, _ = env3.reset(seed=99)
for _ in range(500):
    observations.append(obs.copy())
    concepts["pole_falling_right"].append(int(obs[2] > 0))
    concepts["pole_falling_fast"].append(int(abs(obs[3]) > 0.5))
    concepts["cart_off_centre"].append(int(abs(obs[0]) > 0.5))
    action = q_net(torch.FloatTensor(obs)).argmax().item()
    obs, _, t, tr, _ = env3.step(action)
    if t or tr: obs, _ = env3.reset()
env3.close()

reps = get_hidden_representations(q_net, observations, layer_idx=1)
print("Concept probe: does the policy hidden layer encode interpretable concepts?")
print(f"{'Concept':<25} {'CV Accuracy':>12} {'Baseline':>10}")
print("-"*50)
for concept_name, labels in concepts.items():
    labels = np.array(labels)
    baseline = max(labels.mean(), 1-labels.mean())
    if labels.sum() < 10 or (len(labels)-labels.sum()) < 10: continue
    probe = LogisticRegression(max_iter=500)
    acc = cross_val_score(probe, reps, labels, cv=5).mean()
    print(f"  {concept_name:<23} {acc:>11.3f} {baseline:>10.3f}")


## When to use each explanation method

| Method | Best for | Limitation |
|--------|----------|------------|
| Gradient saliency | Quick debugging | Noisy, can highlight irrelevant features |
| Integrated gradients | Final attribution reports | Slower, baseline choice matters |
| Concept probes | Interpretability audits | Requires defining concepts in advance |
| Counterfactuals | Stakeholder explanations | Requires a generative model of observations |